# Рекуррентные нейросети

Построим простейшую нейросеть для посимвольной генерации текста

In [190]:
import pandas as pd  # для работы с данными
import time  # для оценки времени
import torch  # для написания нейросети

## Загрузка данных

Будем работать с датасетом реплик из Симпсонов. Нам нужно извлечь предобработанные тексты и закодировать их числами

In [191]:
df = pd.read_csv('data.csv')
df.head()

,Unnamed: 0,id,episode_id,number,raw_text,timestamp_in_ms,speaking_line,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
0,0,10368,35,29,"Lisa Simpson: Maggie, look. What's that?",235000,True,9,5.0,Lisa Simpson,Simpson Home,"Maggie, look. What's that?",maggie look whats that,4.0
1,1,10369,35,30,Lisa Simpson: Lee-mur. Lee-mur.,237000,True,9,5.0,Lisa Simpson,Simpson Home,Lee-mur. Lee-mur.,lee-mur lee-mur,2.0
2,2,10370,35,31,Lisa Simpson: Zee-boo. Zee-boo.,239000,True,9,5.0,Lisa Simpson,Simpson Home,Zee-boo. Zee-boo.,zee-boo zee-boo,2.0
3,3,10372,35,33,Lisa Simpson: I'm trying to teach Maggie that ...,245000,True,9,5.0,Lisa Simpson,Simpson Home,I'm trying to teach Maggie that nature doesn't...,im trying to teach maggie that nature doesnt e...,24.0
4,4,10374,35,35,"Lisa Simpson: It's like an ox, only it has a h...",254000,True,9,5.0,Lisa Simpson,Simpson Home,"It's like an ox, only it has a hump and a dewl...",its like an ox only it has a hump and a dewlap...,18.0


In [192]:
phrases = df['normalized_text'].tolist()  # колонка с предобработанными текстами
phrases[:10]

['maggie look whats that',
 'lee-mur lee-mur',
 'zee-boo zee-boo',
 'im trying to teach maggie that nature doesnt end with the barnyard i want her to have all the advantages that i didnt have',
 'its like an ox only it has a hump and a dewlap hump and dew-lap hump and dew-lap',
 'you know his blood type how romantic',
 'oh yeah whats my shoe size',
 'ring',
 'yes dad',
 'ooh look maggie what is that do-dec-ah-edron dodecahedron']

In [193]:
text = [[c for c in ph] for ph in phrases if type(ph) is str]

## Создаём массив с данными

Нужно

1. Разбить данные на токены (у нас символы)
2. Закодировать числами
3. Превратить в эмбеддинги

In [194]:
CHARS = set('abcdefghijklmnopqrstuvwxyz ')  # все символы, которые мы хотим использовать для кодировки = наш словарь
INDEX_TO_CHAR = ['none'] + [w for w in CHARS]  # все неизвестные символы будут получать тег none
CHAR_TO_INDEX = {w: i for i, w in enumerate(INDEX_TO_CHAR)}  # словарь токен-индекс

In [195]:
len(INDEX_TO_CHAR)

28

In [196]:
MAX_LEN = 50  # мы хотим ограничить максимальную длину ввода
X = torch.zeros((len(text), MAX_LEN), dtype=int)  # создаём пустой вектор для текста, чтобы класть в него индексы токенов
for i in range(len(text)):  # для каждого предложения
    for j, w in enumerate(text[i]):  # для каждого токена
        if j >= MAX_LEN:
            break
        X[i, j] = CHAR_TO_INDEX.get(w, CHAR_TO_INDEX['none'])

In [197]:
X[0:5]

tensor([[ 6,  2,  4,  4,  5, 11, 21, 18, 19, 19, 20, 21,  7,  8,  2,  9, 14, 21,
          9,  8,  2,  9,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [18, 11, 11,  0,  6, 24, 23, 21, 18, 11, 11,  0,  6, 24, 23,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 1, 11, 11,  0, 25, 19, 19, 21,  1, 11, 11,  0, 25, 19, 19,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 5,  6, 21,  9, 23, 13,  5, 15,  4, 21,  9, 19, 21,  9, 11,  2, 12,  8,
         21,  6,  2,  4,  4,  5, 11, 21,  9,  8,  2,  9, 21, 15,  2,  9, 24, 23,
         11, 21, 17, 19, 11, 14, 15,  9, 21, 11, 15, 17, 21,  7],
        [ 5,  9, 14, 21, 18,  5, 20, 11, 21,  2, 15, 21, 19, 16, 21, 19, 15, 18,
       

## Embedding и RNN ячейки

Каждому токену мы хотим сопоставить не просто число, но вектор. Поэтому вектор текста нам нужно умножить на матрицу эмбеддингов, которая тоже будет учиться в процессе обучения нейросети. Для создания такой матрицы нам нужен слой `nn.Embedding`

In [198]:
X[0:5].shape

torch.Size([5, 50])

In [199]:
embeddings = torch.nn.Embedding(len(INDEX_TO_CHAR), 28)  # размер словаря * размер вектора для кодировки каждого слова
t = embeddings(X[0:5])
t.shape

torch.Size([5, 50, 28])

In [200]:
t.shape, X[0:5].shape

(torch.Size([5, 50, 28]), torch.Size([5, 50]))

In [201]:
rnn = torch.nn.RNN(28, 128, batch_first=True)  # на вход - размер эмбеддинга, размер скрытого состояния и порядок размерностей
o, s = rnn(t)
# вектора для слов: батч * число токенов * размер скрытого состояния
# вектор скрытого состояния: число вектров (один) * батч * размер скрытого состояния
o.shape, s.shape

(torch.Size([5, 50, 128]), torch.Size([1, 5, 128]))

Можно применять несколько рекуррентных ячеек подряд

In [202]:
o, s2 = rnn(t, s)
o.shape, s2.shape

(torch.Size([5, 50, 128]), torch.Size([1, 5, 128]))

## Реализация сети с RNN
3 слоя:
1. Embeding (30)
2. RNN (hidden_dim=128)
3. Полносвязный слой для предсказания буквы (28, то есть размер словаря)

In [203]:
class RNNCell(torch.nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_xh = torch.nn.Linear(input_size, hidden_size)
        self.W_hh = torch.nn.Linear(hidden_size, hidden_size)

    def forward(self, x, h_prev):
        return torch.tanh(self.W_xh(x) + self.W_hh(h_prev))

In [204]:
class Network(torch.nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        self.embedding = torch.nn.Embedding(28, 30)
        self.rnn_cell = RNNCell(30, 128)
        self.out = torch.nn.Linear(128, 28)

    def forward(self, sentences):
        batch_size, seq_len = sentences.shape
        h = torch.zeros(batch_size, 128)
        outputs = []

        for t in range(seq_len):
            x_t = self.embedding(sentences[:, t])
            h = self.rnn_cell(x_t, h)
            outputs.append(self.out(h))

        return torch.stack(outputs, dim=1)

In [205]:
model = Network()

In [206]:
criterion = torch.nn.CrossEntropyLoss()  # типичный лосс многоклассовой классификации
optimizer = torch.optim.SGD(model.parameters(), lr=.05)

Обучение:

In [207]:
for ep in range(20):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    for i in range(int(len(X) / 100)):
        # берём батч в 100 элементов
        batch = X[i * 100:(i + 1) * 100]
        X_batch = batch[:, :-1]
        Y_batch = batch[:, 1:].flatten()

        optimizer.zero_grad()
        answers = model.forward(X_batch)
        answers = answers.view(-1, len(INDEX_TO_CHAR))
        loss = criterion(answers, Y_batch)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    print("Epoch {}. Time: {:.3f}, Train loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))

Epoch 0. Time: 0.969, Train loss: 2.013
Epoch 1. Time: 0.916, Train loss: 1.753
Epoch 2. Time: 1.052, Train loss: 1.677
Epoch 3. Time: 0.906, Train loss: 1.632
Epoch 4. Time: 0.921, Train loss: 1.601
Epoch 5. Time: 0.903, Train loss: 1.577
Epoch 6. Time: 0.906, Train loss: 1.557
Epoch 7. Time: 0.992, Train loss: 1.541
Epoch 8. Time: 0.945, Train loss: 1.527
Epoch 9. Time: 0.935, Train loss: 1.514
Epoch 10. Time: 0.932, Train loss: 1.502
Epoch 11. Time: 0.936, Train loss: 1.491
Epoch 12. Time: 1.024, Train loss: 1.480
Epoch 13. Time: 0.942, Train loss: 1.470
Epoch 14. Time: 0.942, Train loss: 1.461
Epoch 15. Time: 1.003, Train loss: 1.452
Epoch 16. Time: 0.948, Train loss: 1.444
Epoch 17. Time: 0.947, Train loss: 1.436
Epoch 18. Time: 0.942, Train loss: 1.428
Epoch 19. Time: 0.943, Train loss: 1.421


## Генерация


- Сначала отправлем в модель буквы из предложения (прогревая состояние)
- Затем берём самую вероятную букву и добавляем её в предложение
- Повторяем пока не получим none (0)

In [208]:
CHAR_TO_INDEX['none']

0

In [209]:
def generate_sentence(word, max_new=50, temperature=0.8):
    sentence = list(word)
    input_seq = [CHAR_TO_INDEX.get(s, 0) for s in sentence]

    for _ in range(max_new):
        inp = torch.tensor([input_seq], dtype=torch.long)
        with torch.no_grad():
            logits = model(inp)

        next_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_logits, dim=-1)

        next_idx = torch.multinomial(probs, 1).item()

        if next_idx == 0:
            break

        input_seq.append(next_idx)

    new_chars = [INDEX_TO_CHAR[i] for i in input_seq[len(sentence):]]
    return word + ''.join(new_chars)

In [210]:
generate_sentence('dog')

'doge it wemale srestont rear snot it thes cick is mak'

In [211]:
generate_sentence('It is')

'It iswere juke there sobr thow fnos sare'